# Detección de anomalías sobre el dataset de los docentes

Aplica a un CSV cualquiera las dos técnicas no supervisadas del obligatorio, con el mismo protocolo y los mismos hiperparámetros que `laboratorio_comparacion_detectores.ipynb`:

- `StandardScaler` y modelos se ajustan sobre **todas** las filas. No hay etiquetas para entrenar.
- **Isolation Forest**: `n_estimators=300`, `max_samples=min(256, n)`, `contamination=0.10`, `random_state=42`. Score `-score_samples(X)`.
- **One-Class SVM RBF**: `kernel="rbf"`, `nu=0.10`, `gamma="scale"`. Score `-decision_function(X)`.
- Flag de alerta: `predict(X) == -1`.

`contamination` y `nu` son un **presupuesto de alertas** (cuántas filas puede investigar el equipo), no la tasa real de anomalías.

## Cómo usarlo

Completar la celda de parámetros, o pasarlos por variables de entorno sin tocar el cuaderno:

```bash
DATA_PATH=datos/dataset.csv FEATURES=requests,error_rate,latency_ms,cpu_pct LABEL_COL=is_anomaly \
  jupyter nbconvert --to notebook --execute --output resultado.ipynb deteccion_dataset_docentes.ipynb
```

- `DATA_PATH`: ruta al CSV, relativa a esta carpeta o absoluta.
- `FEATURES`: lista de columnas numéricas. Si queda vacía, se usan todas las columnas numéricas menos `LABEL_COL`.
- `LABEL_COL` (opcional): columna con la etiqueta real. **Solo se usa para evaluar**, nunca para entrenar.

Si `DATA_PATH` no existe, el cuaderno lo avisa y termina sin error.

In [1]:
# Parámetros. Las variables de entorno del mismo nombre tienen prioridad.
import os

DATA_PATH = "datos/dataset_docentes.csv"
FEATURES = []          # vacío = todas las columnas numéricas menos LABEL_COL
LABEL_COL = None       # opcional, solo para evaluar

DATA_PATH = os.environ.get("DATA_PATH", DATA_PATH)
if os.environ.get("FEATURES"):
    FEATURES = [c.strip() for c in os.environ["FEATURES"].split(",") if c.strip()]
LABEL_COL = os.environ.get("LABEL_COL") or LABEL_COL

SEED = 42
CONTAMINATION = 0.10
NU = 0.10

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

TRUE_LABELS = {"1", "true", "t", "yes", "y", "si", "sí", "anomaly", "anomalia", "anomalía", "-1"}

data_file = Path(DATA_PATH)
DATA_OK = data_file.is_file()
if not DATA_OK:
    print(f"DATASET NO ENCONTRADO: {data_file.as_posix()} (ruta relativa a la carpeta del cuaderno)")
    print("Estado: dataset de los docentes pendiente de recepción.")
    print("Copiá el CSV a esa ruta o definí DATA_PATH, y volvé a ejecutar. No se entrenó ningún modelo.")
else:
    raw = pd.read_csv(data_file)
    print(f"Archivo: {data_file}  |  filas: {len(raw)}  |  columnas: {list(raw.columns)}")

    if LABEL_COL is not None and LABEL_COL not in raw.columns:
        raise ValueError(f"LABEL_COL={LABEL_COL!r} no está en el CSV. Columnas: {list(raw.columns)}")
    if not FEATURES:
        FEATURES = [c for c in raw.select_dtypes(include="number").columns if c != LABEL_COL]
    missing = [c for c in FEATURES if c not in raw.columns]
    if missing:
        raise ValueError(f"Columnas de FEATURES que no están en el CSV: {missing}")
    if LABEL_COL in FEATURES:
        raise ValueError("LABEL_COL no puede estar en FEATURES: la etiqueta es solo para evaluar.")
    not_numeric = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(raw[c])]
    if not_numeric:
        raise ValueError(f"Columnas de FEATURES que no son numéricas: {not_numeric}")

    data = raw.dropna(subset=FEATURES).reset_index(drop=True)
    if len(data) < len(raw):
        print(f"Se descartaron {len(raw) - len(data)} filas con valores vacíos en FEATURES.")
    print("FEATURES:", FEATURES, "|  LABEL_COL:", LABEL_COL)
    display(data[FEATURES].describe().round(4))

DATASET NO ENCONTRADO: datos/dataset_docentes.csv (ruta relativa a la carpeta del cuaderno)
Estado: dataset de los docentes pendiente de recepción.
Copiá el CSV a esa ruta o definí DATA_PATH, y volvé a ejecutar. No se entrenó ningún modelo.


In [3]:
if DATA_OK:
    # Entrenamiento no supervisado: solo FEATURES, sobre todas las filas.
    scaler = StandardScaler().fit(data[FEATURES])
    X_all = scaler.transform(data[FEATURES])

    forest = IsolationForest(
        n_estimators=300,
        max_samples=min(256, len(data)),
        contamination=CONTAMINATION,
        random_state=SEED,
    ).fit(X_all)
    data["score_if"] = -forest.score_samples(X_all)
    data["flag_if"] = forest.predict(X_all) == -1

    ocsvm = OneClassSVM(kernel="rbf", nu=NU, gamma="scale").fit(X_all)
    data["score_ocsvm"] = -ocsvm.decision_function(X_all)
    data["flag_ocsvm"] = ocsvm.predict(X_all) == -1

    data["rank_if"] = data["score_if"].rank(ascending=False, method="min").astype(int)
    data["rank_ocsvm"] = data["score_ocsvm"].rank(ascending=False, method="min").astype(int)

    resumen = pd.DataFrame({
        "alertas": [int(data.flag_if.sum()), int(data.flag_ocsvm.sum())],
        "% de filas": [100 * data.flag_if.mean(), 100 * data.flag_ocsvm.mean()],
    }, index=["Isolation Forest", "One-Class SVM"]).round(1)
    print(f"Filas: {len(data)}  |  alertas en ambos: {int((data.flag_if & data.flag_ocsvm).sum())}"
          f"  |  solo IF: {int((data.flag_if & ~data.flag_ocsvm).sum())}"
          f"  |  solo OCSVM: {int((~data.flag_if & data.flag_ocsvm).sum())}")
    display(resumen)

In [4]:
if DATA_OK:
    # Tabla de alertas: toda fila que enciende al menos un detector, ordenada por score IF.
    alert_cols = [c for c in data.columns if c not in FEATURES and not c.startswith(("score_", "flag_", "rank_"))]
    cols = [*alert_cols, *FEATURES, "score_if", "rank_if", "flag_if", "score_ocsvm", "rank_ocsvm", "flag_ocsvm"]
    alertas = data.loc[data.flag_if | data.flag_ocsvm, cols].sort_values("score_if", ascending=False)
    print(f"Tabla de alertas ({len(alertas)} filas):")
    display(alertas.round(4))

In [5]:
def to_bool_label(series: pd.Series) -> pd.Series:
    # bool tal cual; numérico: distinto de 0 es anomalía; texto: valores de TRUE_LABELS.
    if pd.api.types.is_bool_dtype(series):
        return series
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0) != 0
    return series.astype(str).str.strip().str.lower().isin(TRUE_LABELS)


if DATA_OK and LABEL_COL is not None:
    # Evaluación: la etiqueta se mira recién acá, después de entrenar.
    y_true = to_bool_label(data[LABEL_COL])
    rows = {}
    for name, flag in (("Isolation Forest", "flag_if"), ("One-Class SVM", "flag_ocsvm")):
        p, r, f1, _ = precision_recall_fscore_support(y_true, data[flag], average="binary", zero_division=0)
        rows[name] = {"precision": p, "recall": r, "f1": f1, "alertas": int(data[flag].sum())}
    metricas = pd.DataFrame(rows).T
    print(f"Anomalías reales según {LABEL_COL}: {int(y_true.sum())} de {len(data)} ({100 * y_true.mean():.1f}%)")
    display(metricas.round(3))
elif DATA_OK:
    print("Sin LABEL_COL: no se calculan precisión, recall ni F1. La tabla de alertas es el resultado.")

In [6]:
if DATA_OK:
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for ax, score, flag, title in (
        (axes[0], "score_if", "flag_if", "Isolation Forest"),
        (axes[1], "score_ocsvm", "flag_ocsvm", "One-Class SVM RBF"),
    ):
        ax.plot(data.index, data[score], color="#183B66", lw=1.0)
        ax.scatter(data.index[data[flag]], data.loc[data[flag], score], c="#E76F51", s=16, zorder=3, label="alerta")
        ax.set_ylabel(f"score {title}")
        ax.legend(loc="upper left")
    axes[1].set_xlabel("fila del CSV")
    axes[0].set_title("Scores por fila (más alto = más anómalo)")
    plt.tight_layout()
    plt.show()